# Laboratorio de regresión logística

|                |   |
:----------------|---|
| **Nombre**     |  Diego Dueñas Martín |
| **Fecha**      |   16/03/2026|
| **Expediente** |   751426| 

In machine learning, Support Vector Machines (SVM) are supervised learning models with associated learning algorithms that analyze data used for classification and regression analysis. It is mostly used in classification problems. In this algorithm, each data item is plotted as a point in p-dimensional space (where p is the number of features), with the value of each feature being the value of a particular coordinate. Then, classification is performed by finding the hyper-plane that best differentiates the two classes (or more if we have a multi class problem):

$$ f(x) = w^T \varphi(x) + b $$

where $\varphi: X \rightarrow F $ is a function that makes each input point $x$ correspond to a point in F, where F is a Hilbert space.

In addition to performing linear classification, SVMs can efficiently perform a non-linear classification, implicitly mapping their inputs into high-dimensional feature spaces (more specifically using the kernel trick, like the RBF funcion). 

[1]

OLS utilizes the squared residuals to fit the parameters. Large residuals caused by outliers may worsen the accuracy significantly.

Support Vectors use piecewise linear functions to counter this, in which a hyperparameter  $\epsilon$ called the margin lets errors that are less or equal to it be 0, and error larger than it be $e - \epsilon$. 

The problem to solve is:

\begin{split}
        \min_{w, b, \xi, \xi^*} \mathcal{P}_\epsilon(w, b, \xi) &= \frac{1}{2} w^T w + c \sum_{k=1}^{N} \xi_k \\
        \text{s.t. } & y_k [w^T \varphi(x_k) - b] \geq 1- \xi_k,\ \ k = 1, ..., N \\
        & \xi_k \geq 0,\ \ k = 1, ..., N
\end{split}


The most important question that arises when using a SVM is how to choose the correct hyperplane. Consider the following scenarios:

### Scenario 1

In this scenario there are three hyperplanes called A, B, and C. Now, the problem is to identify the hyperplane which best differentiates the stars and the circles.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_21-2.png" alt="what image shows"></center>

In this case, hyperplane B separates the stars and the circle betters, hence it is the correct hyperplane.


### Scenario 2

Now take another scenario where all three hyperplanes are segregating classes well. The question that arises is how to choose the best hyperplane in this situation.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_4-2.png" alt="what image shows"></center>

In such scenarios, we calculate the margin (which is the distance between nearest data point and the hyperplane). The hyperplane with the largest margin will be considered as the correct hyperplane to classify the dataset.

Here C has the largest margin. Hence, it is considered as the best hyperplane.


### Kernels
Knowing 
$$ w = \sum_{k=1}^{N} \alpha_k y_k \varphi(x_k) $$

And
$$ y_{pred} = w^T \varphi(x) + b $$

Then 
$$ y_{pred} = (\sum_{k=1}^{N} \alpha_k y_k \varphi(x_k))^T \varphi(x) + b $$

Where $\varphi$ is a function that makes each input in $x$ correspond to a point in $F$ (a Hilbert space). This can be seen as processing and transforming the input featuers to keep the model's convexity. [2]

This also allows us to transform the inputs into another space where they might be more easily classified.

<center><img src=https://miro.medium.com/max/838/1*gXvhD4IomaC9Jb37tzDUVg.png alt="what image shows"></center>

## ROC and AUC

A ROC (Receiver Operating Characteristic) is a graph that shows how the classification model performs at the classification thresholds. 

ROC curves typically feature true positive rate on the Y axis, and false positive rate on the X axis. This means that the top left corner of the plot is the “ideal” point - a false positive rate of zero, and a true positive rate of one. This is not very realistic, but it does mean that a larger area under the curve (AUC) is usually better. [3]

True Positive Rate is a synonym for Recall and defined as:
$$ TPR = \frac{TP}{TP + FN} $$

False Positive Rate is a synonym for Specificity and defined as:

$$ FPR = \frac{FP}{FP + TN} $$

ROC curves are typically used in binary classification to study the output of a classifier. In order to extend ROC curve and ROC area to multi-label classification, it is necessary to binarize the output. One ROC curve can be drawn per label, but one can also draw a ROC curve by considering each element of the label indicator matrix as a binary prediction (micro-averaging).

E.g. If you lower a classification threshold, more items would be classified as positive, increasing False Positives and True Positives.

AUC stands for Area under the ROC.

## Ejercicio 1

- Utiliza el dataset `Iris`, modela con SVC y haz Cross-Validation de diferentes kernels ('linear', 'poly', 'rbf', 'sigmoid').
- Modela con LogisticRegression.
- El método de Cross-Validation es K-Folds con $k=10$.
- Utiliza el AUC como métrico de Cross-Validation.
- Compara resultados.

In [7]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
resultados_iris = {}

for kernel in kernels:
    svc = SVC(kernel=kernel, probability=True, random_state=42)
    scores = cross_val_score(svc, X_iris, y_iris, cv=cv, scoring='roc_auc_ovr')
    resultados_iris[f'SVC ({kernel})'] = scores

lr = LogisticRegression(max_iter=1000, random_state=42)
resultados_iris['LogisticRegression'] = cross_val_score(lr, X_iris, y_iris, cv=cv, scoring='roc_auc_ovr')
print(f"{'Modelo':<25} {'AUC Media':>10} {'Desv. Std':>10}")

for modelo, scores in resultados_iris.items():
    print(f"{modelo:<25} {scores.mean():>10.4f} {scores.std():>10.4f}")

# rbf es el kernel más robusto en Iris y supera ligeramente al resto.
# sigmoid se desempeña notablemente peor; no es adecuado para este dataset.
# LogisticRegression iguala a linear y poly, lo que confirma que el problema
# es casi linealmente separable.


Modelo                     AUC Media  Desv. Std
SVC (linear)                  0.9967     0.0068
SVC (poly)                    0.9987     0.0040
SVC (rbf)                     0.9987     0.0040
SVC (sigmoid)                 0.9753     0.0273
LogisticRegression            0.9980     0.0060


## Ejercicio 2
- Repite el ejercicio 1 con el dataset `Default`. Utiliza `default` como target.

In [ ]:
data = pd.read_csv('Default.csv')
data['student'] = (data['student'] == 'Yes').astype(int)
y_def = (data['default'] == 'Yes').astype(int).values
X_def = StandardScaler().fit_transform(data[['student', 'balance', 'income']])

rng = np.random.default_rng(42)
idx = rng.choice(len(y_def), 300, replace=False)
X_svc, y_svc = X_def[idx], y_def[idx]

cv_def = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Default es binario 
resultados_def = {}

for kernel in kernels:
    svc = SVC(kernel=kernel, probability=True, random_state=42)
    scores = cross_val_score(svc, X_svc, y_svc, cv=cv_def, scoring='roc_auc')
    resultados_def[f'SVC ({kernel})'] = scores

# LogisticRegression sí corre sobre el dataset completo
lr_def = LogisticRegression(max_iter=1000, random_state=42)
resultados_def['LogisticRegression (full)'] = cross_val_score(
    lr_def, X_def, y_def, cv=cv_def, scoring='roc_auc'
)

print(f"{'Modelo':<28} {'AUC Media':>10} {'Desv. Std':>10}")
for modelo, scores in resultados_def.items():
    print(f"{modelo:<28} {scores.mean():>10.4f} {scores.std():>10.4f}")

# LogisticRegression sobre el dataset completo obtiene el AUC más alto y estable.
# Los kernels SVC sobre la muestra de 300 muestran más varianza; linear y sigmoid
# se comportan mejor que rbf y poly, lo que indica que la frontera de decisión
# en Default es esencialmente lineal (dominada por la columna balance).

Modelo                        AUC Media  Desv. Std
SVC (linear)                     0.9309     0.1090
SVC (poly)                       0.8189     0.3337
SVC (rbf)                        0.6799     0.3593
SVC (sigmoid)                    0.9340     0.0752
LogisticRegression (full)        0.9485     0.0221


# Addendum

Métricos disponibles para clasificación:
- ‘accuracy’
- ‘balanced_accuracy’
- ‘top_k_accuracy’
- ‘average_precision’
- ‘neg_brier_score’
- ‘f1’
- ‘f1_micro’
- ‘f1_macro’
- ‘f1_weighted’
- ‘f1_samples’
- ‘neg_log_loss’
- ‘precision’ etc.
- ‘recall’ etc.
- ‘jaccard’ etc.
- ‘roc_auc’
- ‘roc_auc_ovr’
- ‘roc_auc_ovo’
- ‘roc_auc_ovr_weighted’
- ‘roc_auc_ovo_weighted’
- ‘d2_log_loss_score’

# References

[1] Shigeo Abe.Support Vector Machines for Pattern Classification,2Ed.Springer-Verlag London,2010. ISBN978-1-84996-097-7. URLhttps://www.springer.com/gp/book/9781849960977.

[2] Johan A K Suykens, Tony Van Gestel, Jos De Brabanter, BartDe Moor, and Joos Vandewalle.Least Squares Support VectorMachines. World Scientific,2002. ISBN9789812381514. URLhttps://www.worldscientific.com/worldscibooks/10.1142/5089.

[3] Bradley, A. P. (1997). The use of the area under the ROC curve in the evaluation of machine learning algorithms. Pattern recognition, 30(7), 1145-1159. URL https://www.researchgate.net/post/how_can_I_interpret_the_ROC_curve_result